# Superdense coding

Inverso do teletransporte envia 2 bits clássicos usando apenas 1 Qubit

É a demonstração mais direta de que um canal quântico tem capacidae maior que um canal clássico

|BITS| GATE ALICE | ESTADO RESULTANTE                | ESTADO DE BELL |
|----|------------|----------------------------------|---------------|
| 00 | \|(nada) | (\|00⟩ + \|11⟩) / √2             | \|Φ⁺⟩         |
|01 | X| (\|01⟩ +             \|10⟩) / √2 | \|Ψ⁺⟩         |
|10 | Z| (\|00⟩ − \|11⟩) / √2             | \|Φ⁻⟩         |
|11|XZ (ou iY)| (\|01⟩ − \|10⟩) / √2 | \|Ψ⁻⟩         |

Na tabela de enconding cada par de bits clássicos (00, 01, 10, 11) corresponde a um dos 4 estados de Bell - e como eles são ortogonaris, Bob pode distingui-los perfeitamente com uma medição de Bell.

### Código completo nos dois frameworks

In [1]:
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_aer import AerSimulator

def superdense_encode(bits: str) -> QuantumCircuit:
    """
    bits: string '00', '01', '10' ou '11'
    Retorna circuito com codificação de Alice.
    """
    b0, b1 = int(bits[0]), int(bits[1])

    qr = QuantumRegister(2, 'q')
    cr = ClassicalRegister(2, 'c')
    qc = QuantumCircuit(qr, cr)

    # ── Fase 0: criar par de Bell |Φ⁺⟩ (normalmente pré-compartilhado)
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier(label="par de Bell criado")

    # ── Fase 1: Alice codifica seus 2 bits em q0
    # b1=1 → aplica X antes do Z (flip de amplitude)
    # b0=1 → aplica Z (flip de fase)
    if b1 == 1:
        qc.x(0)
    if b0 == 1:
        qc.z(0)
    qc.barrier(label=f"Alice codificou {bits}")

    # ── Fase 2: Bob decodifica (Bell measurement em q0 e q1)
    qc.cx(0, 1)
    qc.h(0)
    qc.barrier(label="Bob decodificou")

    # ── Fase 3: Medir
    qc.measure(qr, cr)
    return qc


# ── Testar todos os 4 casos ────────────────────────────────────────────
sim = AerSimulator()

print("Bits enviados → Bits recebidos")
print("─" * 34)

for bits in ["00", "01", "10", "11"]:
    qc = superdense_encode(bits)
    job = sim.run(qc, shots=1024)
    counts = job.result().get_counts()
    # resultado esperado: 100% no estado = bits enviados
    received = max(counts, key=counts.get)
    ok = "✓" if received == bits[::-1] else "✗"  # Qiskit inverte bit order
    print(f"  {bits}  →  {received[::-1]}  {ok}  {counts}")

Bits enviados → Bits recebidos
──────────────────────────────────
  00  →  00  ✓  {'00': 1024}
  01  →  01  ✓  {'10': 1024}
  10  →  10  ✓  {'01': 1024}
  11  →  11  ✓  {'11': 1024}


## Visualizar o circuito

In [2]:
qc = superdense_encode("11")
print(qc.draw('text'))

     ┌───┐      par de Bell criado ┌───┐┌───┐ Alice codificou 11      ┌───┐»
q_0: ┤ H ├──■───────────░──────────┤ X ├┤ Z ├─────────░────────────■──┤ H ├»
     └───┘┌─┴─┐         ░          └───┘└───┘         ░          ┌─┴─┐└───┘»
q_1: ─────┤ X ├─────────░─────────────────────────────░──────────┤ X ├─────»
          └───┘         ░                             ░          └───┘     »
c: 2/══════════════════════════════════════════════════════════════════════»
                                                                           »
«      Bob decodificou ┌─┐   
«q_0: ────────░────────┤M├───
«             ░        └╥┘┌─┐
«q_1: ────────░─────────╫─┤M├
«             ░         ║ └╥┘
«c: 2/══════════════════╩══╩═
«                       0  1 


### Cirq (mesma lógica, sintaxe diferente)

In [ ]:
import cirq
import numpy as np

def superdense_cirq(bits: str):
    b0, b1 = int(bits[0]), int(bits[1])

    q0, q1 = cirq.LineQubit.range(2)
    circuit = cirq.Circuit()

    # Criar par de Bell
    circuit.append([
        cirq.H(q0),
        cirq.CNOT(q0, q1),
    ])

    # Alice codifica
    alice_ops = []
    if b1 == 1:
        alice_ops.append(cirq.X(q0))
    if b0 == 1:
        alice_ops.append(cirq.Z(q0))
    if alice_ops:
        circuit.append(alice_ops)

    # Bob decodifica
    circuit.append([
        cirq.CNOT(q0, q1),
        cirq.H(q0),
    ])

    # Medir
    circuit.append([
        cirq.measure(q0, key='b0'),
        cirq.measure(q1, key='b1'),
    ])

    return circuit


# ── Simular ────────────────────────────────────────────────────────────
sim = cirq.Simulator()

print("Bits enviados → Bits recebidos")
print("─" * 34)

for bits in ["00", "01", "10", "11"]:
    circuit = superdense_cirq(bits)
    result = sim.run(circuit, repetitions=1024)

    b0_counts = result.measurements['b0'].flatten()
    b1_counts = result.measurements['b1'].flatten()

    # pegar o resultado mais frequente
    from collections import Counter
    b0_val = Counter(b0_counts).most_common(1)[0][0]
    b1_val = Counter(b1_counts).most_common(1)[0][0]
    received = f"{b0_val}{b1_val}"

    ok = "✓" if received == bits else "✗"
    print(f"  {bits}  →  {received}  {ok}")

# Imprimir o circuito para '11'
print("\nCircuito para '11':")
print(superdense_cirq("11"))